# Basic Analysis — Questions 1-5

**Techniques:** `GROUP BY`, `JOIN`, `ORDER BY`, `CASE WHEN`, `AVG`

| # | Question |
|---|----------|
| Q1 | Top product categories by order volume |
| Q2 | Distribution of order statuses |
| Q3 | States with the most customers |
| Q4 | Average order value by payment type |
| Q5 | Average review score by product category |

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.db_utils import run_query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Q1: Top 15 Product Categories by Order Volume

In [ ]:
q1 = run_query("""
    SELECT
        t.product_category_name_english AS category,
        COUNT(oi.order_id)              AS total_orders
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_name_translation t
        ON p.product_category_name = t.product_category_name
    GROUP BY category
    ORDER BY total_orders DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=q1, x='total_orders', y='category', ax=ax, color='steelblue')
ax.set_title('Top 15 Product Categories by Order Volume', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Orders')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../data/q1_top_categories.png', bbox_inches='tight')
plt.show()
q1

## Q2: Order Status Distribution

In [ ]:
q2 = run_query("""
    SELECT
        order_status,
        COUNT(*) AS total_orders,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
    FROM orders
    GROUP BY order_status
    ORDER BY total_orders DESC
""")

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(q2['total_orders'], labels=q2['order_status'], autopct='%1.1f%%', startangle=140)
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/q2_order_status.png', bbox_inches='tight')
plt.show()
q2

## Q3: Top 10 States by Customer Count

In [ ]:
q3 = run_query("""
    SELECT
        c.customer_state        AS state,
        COUNT(DISTINCT c.customer_id) AS unique_customers
    FROM customers c
    GROUP BY state
    ORDER BY unique_customers DESC
    LIMIT 10
""")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=q3, x='state', y='unique_customers', ax=ax, color='coral')
ax.set_title('Top 10 States by Customer Count', fontsize=14, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Unique Customers')
plt.tight_layout()
plt.savefig('../data/q3_customers_by_state.png', bbox_inches='tight')
plt.show()
q3

## Q4: Average Order Value by Payment Type

In [ ]:
q4 = run_query("""
    SELECT
        payment_type,
        COUNT(DISTINCT order_id)        AS total_orders,
        ROUND(AVG(payment_value), 2)    AS avg_order_value,
        ROUND(SUM(payment_value), 2)    AS total_revenue
    FROM order_payments
    GROUP BY payment_type
    ORDER BY total_revenue DESC
""")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=q4, x='payment_type', y='avg_order_value', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Avg Order Value by Payment Type')
axes[0].set_xlabel('')

sns.barplot(data=q4, x='payment_type', y='total_revenue', ax=axes[1], color='mediumpurple')
axes[1].set_title('Total Revenue by Payment Type')
axes[1].set_xlabel('')

plt.suptitle('Payment Type Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/q4_payment_analysis.png', bbox_inches='tight')
plt.show()
q4

## Q5: Average Review Score by Category (min 50 reviews)

In [ ]:
q5 = run_query("""
    SELECT
        t.product_category_name_english AS category,
        COUNT(r.review_id)              AS total_reviews,
        ROUND(AVG(r.review_score), 2)   AS avg_score
    FROM order_reviews r
    JOIN orders o       ON r.order_id = o.order_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    JOIN product_category_name_translation t
        ON p.product_category_name = t.product_category_name
    GROUP BY category
    HAVING total_reviews > 50
    ORDER BY avg_score DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if s >= 4.0 else '#e74c3c' for s in q5['avg_score']]
sns.barplot(data=q5, x='avg_score', y='category', ax=ax, palette=colors)
ax.axvline(x=4.0, color='gray', linestyle='--', linewidth=1, label='Score = 4.0')
ax.set_title('Top 15 Categories by Avg Review Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Score (1-5)')
ax.set_ylabel('')
ax.legend()
plt.tight_layout()
plt.savefig('../data/q5_review_by_category.png', bbox_inches='tight')
plt.show()
q5